# 14. Stacks

After this notebook you'll be able to use a list as a **stack**: put things on top, take them off the top, and use that to undo typing, match brackets, and clean up strings.

Like two pointers, this isn't new Python. It's a way of using a list you already know. The Python is small; the hard part is seeing **when** a stack is the right tool. So this notebook draws the stack at every step.

## A pile of plates

Think of a pile of clean plates. You put a new plate **on top**. When you need one, you take it **from the top**. You never pull one out of the middle. The last plate you put on is the first one you take off.

That's a **stack**. In Python, a plain list works as one. The **end** of the list is the top of the pile.

| Plates | Stack word | Python |
| --- | --- | --- |
| put a plate on top | **push** | `stack.append(x)` |
| take the top plate off | **pop** | `stack.pop()` |
| look at the top plate | **peek** | `stack[-1]` |
| no plates left? | **empty** | `len(stack) == 0` |

In [ ]:
plates = []
plates.append("red")
plates.append("blue")
plates.append("green")
print(plates)
print(plates[-1])

top = plates.pop()
print(top)
print(plates)
print(len(plates) == 0)

```
['red', 'blue', 'green']
green
green
['red', 'blue']
False
```

- `append` puts `"green"` at the end, which is the top.
- `plates[-1]` **looks** at the top plate. It doesn't take it off.
- `plates.pop()` **takes** the top plate off and gives it back to you. Here it went into `top`.

**Predict, then run.** Write down what the stack looks like after each line, then run the cell.

In [ ]:
stack = []
stack.append(1)
stack.append(2)
stack.pop()
stack.append(3)
stack.append(4)
stack.pop()
print(stack)

```
[1, 3]
```

| Line | Stack after (top is on the right) |
| --- | --- |
| `append(1)` | `[1]` |
| `append(2)` | `[1, 2]` |
| `pop()` | `[1]` (2 came off) |
| `append(3)` | `[1, 3]` |
| `append(4)` | `[1, 3, 4]` |
| `pop()` | `[1, 3]` (4 came off) |

## An empty stack: check before you take

You can't take a plate from an empty pile. Run this:

In [ ]:
stack = []
stack.pop()

```
IndexError: pop from empty list
```

Peeking at an empty stack breaks too: `stack[-1]` gives `IndexError: list index out of range`.

So before you `pop` or peek, check that the stack isn't empty. There are two ways to write the same check:

In [ ]:
stack = []
print(len(stack) == 0)
print(not stack)

stack.append("tea")
print(len(stack) == 0)
print(not stack)

```
True
True
False
False
```

An empty list counts as `False`, so `not stack` is `True` when it's empty. You'll see `if not stack:` and `if stack:` (meaning "if it has something in it") a lot in other people's code. Both are fine. Use whichever reads better to you.

### The order inside `and` matters

You often want "the stack isn't empty **and** the top is a `(`". Python reads an `and` from left to right, and if the left part is already `False`, it **doesn't look at the right part at all**.

So put the empty check **first**. Run both lines and compare.

In [ ]:
stack = []
print(len(stack) > 0 and stack[-1] == "(")

```
False
```

The left part is `False` (the stack is empty), so Python never tried `stack[-1]`. Now the other way round:

In [ ]:
stack = []
print(stack[-1] == "(" and len(stack) > 0)

```
IndexError: list index out of range
```

Same two questions, but Python asked `stack[-1]` first, on an empty stack, and crashed. **Empty check first, then peek.**

## Stack job 1: the backspace key

When you type, each letter goes on the end. **Backspace** removes the **last** letter you typed. That's a stack: letters are pushed, backspace pops.

Let `#` stand for backspace. Typing `"cat#ke"` should give `"cake"`. Watch the stack:

In [ ]:
def type_out(keys):
    stack = []
    for ch in keys:
        if ch == "#":
            if stack:
                stack.pop()
        else:
            stack.append(ch)
        print(ch, " ", stack)
    return "".join(stack)

print(type_out("cat#ke"))

```
c   ['c']
a   ['c', 'a']
t   ['c', 'a', 't']
#   ['c', 'a']
k   ['c', 'a', 'k']
e   ['c', 'a', 'k', 'e']
cake
```

| Key | What happens | Stack after |
| --- | --- | --- |
| `c` | push | `c` |
| `a` | push | `c a` |
| `t` | push | `c a t` |
| `#` | pop the `t` | `c a` |
| `k` | push | `c a k` |
| `e` | push | `c a k e` |

At the end, `"".join(stack)` glues the letters into a string (notebook 06).

The `if stack:` before `pop` handles pressing backspace when nothing is typed yet. Nothing happens, like on a real keyboard.

**Try it:** run `type_out("##hi")` and `type_out("ab###c")`. Predict first.

## Stack job 2: matching brackets

When are brackets correct? Every opening bracket needs a closing bracket, and they close in the right order: **the most recently opened one closes first**.

```
   ( [ ] )      ✓   [ opened last, so ] must come first
   ( [ ) ]      ✗   ) came while [ was still open
```

"Most recent first" is exactly what a stack does. The plan:

- Opening bracket: **push** it. It's waiting for its partner.
- Closing bracket: the top of the stack must be its partner. If it is, **pop** it (that pair is done). If it isn't, or the stack is empty, the string is wrong.
- At the end, the stack must be **empty**. Anything left was never closed.

Here it is for round brackets only. Run it.

In [ ]:
def round_ok(s):
    stack = []
    for ch in s:
        if ch == "(":
            stack.append(ch)
        else:
            if not stack:
                print(ch, " nothing to close: not valid")
                return False
            stack.pop()
        print(ch, " ", stack)
    return len(stack) == 0

print(round_ok("(()())"))
print()
print(round_ok("(()"))
print()
print(round_ok("())("))

```
(   ['(']
(   ['(', '(']
)   ['(']
(   ['(', '(']
)   ['(']
)   []
True

(   ['(']
(   ['(', '(']
)   ['(']
False

(   ['(']
)   []
)  nothing to close: not valid
False
```

Three ways it can end:

| String | What happened | Answer |
| --- | --- | --- |
| `(()())` | every `)` found a `(` on top, stack empty at the end | `True` |
| `(()` | the loop finished, but one `(` is still on the stack | `False` |
| `())(` | a `)` arrived when the stack was empty | `False` |

### Three kinds of bracket

With `(`, `[` and `{`, a closing bracket has to find **its own** partner on top. A dictionary (notebook 10) is a neat way to say who the partner is:

In [ ]:
partner = {")": "(", "]": "[", "}": "{"}
print(partner["]"])
print("[" in "([{")
print("]" in "([{")

```
[
True
False
```

`ch in "([{"` asks "is `ch` an opening bracket?".

Walk through `"([])"` and `"([)]"` on paper with this table before you write the code in Task 6:

| `ch` | Kind | Top of stack | What happens | Stack after |
| --- | --- | --- | --- | --- |
| `(` | opening | | push | `(` |
| `[` | opening | | push | `( [` |
| `]` | closing | `[` = `partner["]"]` | pop | `(` |
| `)` | closing | `(` = `partner[")"]` | pop | (empty) ✓ |

| `ch` | Kind | Top of stack | What happens | Stack after |
| --- | --- | --- | --- | --- |
| `(` | opening | | push | `(` |
| `[` | opening | | push | `( [` |
| `)` | closing | `[`, but `partner[")"]` is `(` | not partners: `False` ✗ | |

## Stack job 3: pairs that cancel out

Some problems say "when two neighbours are the same, they disappear". After they disappear, the letters on either side become neighbours, and they might cancel too.

`"abbaca"`: the two `b`s cancel, giving `"aaca"`. Now the two `a`s are neighbours, and they cancel too, giving `"ca"`.

A stack handles the "and then the new neighbours" part for free. Compare each letter with the **top** of the stack: the same letter? Pop it (both disappear). Different? Push it.

In [ ]:
def cancel_pairs(s):
    stack = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()
        else:
            stack.append(ch)
        print(ch, " ", stack)
    return "".join(stack)

print(cancel_pairs("abbaca"))

```
a   ['a']
b   ['a', 'b']
b   ['a']
a   []
c   ['c']
a   ['c', 'a']
ca
```

Look at the fourth line. When the second `b` popped the first `b`, the top became `a` again. So the next `a` met it and they cancelled. That's the "new neighbours" part, with no extra work.

Notice `if stack and stack[-1] == ch`: empty check first, then peek.

## Common mistakes

**Popping from an empty stack.** You saw it above: `IndexError: pop from empty list`. In bracket problems, a closing bracket on an empty stack means "not valid", so `return False`. It's not a crash you should let happen.

**Forgetting the end check.** This version forgets that the stack must be empty at the end. Run it:

In [ ]:
def round_ok_wrong(s):
    stack = []
    for ch in s:
        if ch == "(":
            stack.append(ch)
        else:
            if not stack:
                return False
            stack.pop()
    return True

print(round_ok_wrong("(()"))

```
True
```

Wrong answer, no error. `"(("` and `"(()"` have brackets that were never closed. Always finish with `return len(stack) == 0`.

**Using `pop(0)` instead of `pop()`.** `pop(0)` takes the **first** item, the bottom plate. A stack only ever works at the top, so it's always `pop()` with nothing inside.

---

# ✏️ Your turn

Write your code, run your cell, then run the check cell. Each line prints ✓ or ✗.

Put a `print(ch, stack)` inside your loop while you work, like the examples. It shows you exactly where things go wrong.

### Task 1: peek safely

Write `top_or_none(stack)`. It returns the top item of the stack, or `None` if the stack is empty. Don't change the stack.

In [ ]:
def top_or_none(stack):
    # your code here
    pass

In [ ]:
from checks import check

check(top_or_none([1, 2, 3]), 3)
check(top_or_none(["tea"]), "tea")
check(top_or_none([]), None)

s = [4, 5]
top_or_none(s)
check(s, [4, 5])

### Task 2: follow the orders

`orders` is a list of strings. Each one is either `"push N"` (put the number `N` on the stack) or `"pop"` (take the top off). Write `run_orders(orders)`. It starts with an empty stack, follows the orders, and returns the stack at the end. There's never a `"pop"` on an empty stack.

`["push 3", "push 5", "pop", "push 7"]` gives `[3, 7]`.

Hint: `"push 3".split()` gives `["push", "3"]`, and `int("3")` gives `3`.

In [ ]:
def run_orders(orders):
    # your code here
    pass

In [ ]:
check(run_orders(["push 3", "push 5", "pop", "push 7"]), [3, 7])
check(run_orders(["push 10", "push 20", "push 30"]), [10, 20, 30])
check(run_orders(["push 1", "pop"]), [])
check(run_orders([]), [])

### Task 3: the backspace key

Write `type_out(keys)` yourself, without looking back. `#` is backspace. Return the text that's left. Backspace with nothing typed does nothing.

In [ ]:
def type_out(keys):
    # your code here
    pass

In [ ]:
check(type_out("teaa#"), "tea")
check(type_out("ab##"), "")
check(type_out("#a#b"), "b")
check(type_out("hello"), "hello")
check(type_out(""), "")

### Task 4: Backspace String Compare (LeetCode 844)

You get two strings `s` and `t` typed with backspaces (`#`). Return `True` if they end up as the same text.

`s = "ab#c"`, `t = "ad#c"`: both become `"ac"`, so `True`.

Hint: inside the class you can write a small helper function, or copy your `type_out` loop.

In [ ]:
class Solution:
    def backspaceCompare(self, s: str, t: str) -> bool:
        # your code here
        pass

In [ ]:
check(Solution().backspaceCompare("ab#c", "ad#c"), True)
check(Solution().backspaceCompare("ab##", "c#d#"), True)
check(Solution().backspaceCompare("a#c", "b"), False)
check(Solution().backspaceCompare("xy#z", "xzz#"), True)

### Task 5: Maximum Nesting Depth of the Parentheses (LeetCode 1614)

`s` has correct round brackets, plus digits and signs. The **depth** at any moment is how many `(` are open. Return the biggest depth you ever reach.

`"(1+(2*3)+((8)/4))+1"` gives `3`: the `8` sits inside three open brackets.

Hint: push on `(`, pop on `)`, and after each push, look at `len(stack)`. (You could use a counter instead of a stack, and that's fine too.)

In [ ]:
class Solution:
    def maxDepth(self, s: str) -> int:
        # your code here
        pass

In [ ]:
check(Solution().maxDepth("(1+(2*3)+((8)/4))+1"), 3)
check(Solution().maxDepth("(1)+((2))+(((3)))"), 3)
check(Solution().maxDepth("()(())((()()))"), 3)
check(Solution().maxDepth("1+2"), 0)

### Task 6: Valid Parentheses (LeetCode 20)

`s` has only the brackets `( ) [ ] { }`. Return `True` if every bracket is closed by the same kind, in the right order.

Use the `partner` dictionary and the tables from "Three kinds of bracket". There are three ways to be wrong: a closing bracket on an empty stack, a closing bracket whose partner isn't on top, and brackets left open at the end.

In [ ]:
class Solution:
    def isValid(self, s: str) -> bool:
        # your code here
        pass

In [ ]:
check(Solution().isValid("()"), True)
check(Solution().isValid("()[]{}"), True)
check(Solution().isValid("([])"), True)
check(Solution().isValid("{[()()]}"), True)
check(Solution().isValid("(]"), False)
check(Solution().isValid("([)]"), False)
check(Solution().isValid("(("), False)
check(Solution().isValid("]"), False)

### Task 7: Remove All Adjacent Duplicates In String (LeetCode 1047)

Keep removing two neighbouring letters that are the same, until no such pair is left. Return what remains.

`"abbaca"` gives `"ca"`, like `cancel_pairs` above. Write it without looking.

In [ ]:
class Solution:
    def removeDuplicates(self, s: str) -> str:
        # your code here
        pass

In [ ]:
check(Solution().removeDuplicates("abbaca"), "ca")
check(Solution().removeDuplicates("azxxzy"), "ay")
check(Solution().removeDuplicates("aa"), "")
check(Solution().removeDuplicates("abc"), "abc")

### Task 8: Baseball Game (LeetCode 682)

You keep score for a strange baseball game. `operations` is a list of strings, handled in order:

- a number, like `"5"` or `"-2"`: a new score of that many points,
- `"+"`: a new score that's the sum of the **last two** scores,
- `"D"`: a new score that's **double** the last score,
- `"C"`: the last score was a mistake; remove it.

Return the total of all the scores left at the end.

`["5", "2", "C", "D", "+"]`:

| Op | Scores after |
| --- | --- |
| `"5"` | `[5]` |
| `"2"` | `[5, 2]` |
| `"C"` | `[5]` |
| `"D"` | `[5, 10]` |
| `"+"` | `[5, 10, 15]` |

Total: `30`.

Hint: check for `"+"`, `"D"` and `"C"` first. Anything else is a number: `int("-2")` gives `-2`. The last two scores are `scores[-1]` and `scores[-2]`.

In [ ]:
class Solution:
    def calPoints(self, operations: list[str]) -> int:
        # your code here
        pass

In [ ]:
check(Solution().calPoints(["5", "2", "C", "D", "+"]), 30)
check(Solution().calPoints(["5", "-2", "4", "C", "D", "9", "+", "+"]), 27)
check(Solution().calPoints(["1", "C"]), 0)

---

# ✅ Quiz: check yourself

Put your letter between the quotes, then run the cell under it.

**Question 1.** Start with an empty stack. Push `"a"`, push `"b"`, pop, push `"c"`. What's on the stack (top on the right)?

- A) `['a', 'c']`
- B) `['c']`
- C) `['a', 'b', 'c']`

In [ ]:
answer_1 = "?"

In [ ]:
from checks import check_answer
check_answer("14-q1", answer_1)

**Question 2.** Why write `if stack and stack[-1] == "("` and not `if stack[-1] == "(" and stack`?

- A) Python checks the left part first. If the stack is empty, it stops there and never tries `stack[-1]`.
- B) It runs faster, but both are safe.
- C) There's no difference.

In [ ]:
answer_2 = "?"

In [ ]:
check_answer("14-q2", answer_2)

**Question 3.** For `"(()"`, the loop finishes with no problem. What tells you it's **not** valid?

- A) The stack is empty at the end.
- B) A `"("` is still on the stack at the end.
- C) The string starts with `"("`.

In [ ]:
answer_3 = "?"

In [ ]:
check_answer("14-q3", answer_3)

**Question 4.** Which item does `stack.pop()` take?

- A) The first item that was added
- B) A random item
- C) The last item that was added

In [ ]:
answer_4 = "?"

In [ ]:
check_answer("14-q4", answer_4)

---

## Done

You can now:

- use a list as a stack with `append`, `pop` and `[-1]`,
- check that a stack isn't empty before you take from it, with the empty check first in an `and`,
- spot stack problems: undo/backspace, matching brackets, and pairs that cancel out.

Stuck on a task? The answers are in `solutions/14-stacks.ipynb`. Try for real first.

**Next:** lists of lists, to work with rows and columns (`15-grids`).